# Práctica · Semana 8, parte C (opcional): Auditoría de sesgo de un modelo

**Curso:** Introducción a la Inteligencia Artificial · UDES
**Tiempo estimado:** 45 minutos
**Relación con el capítulo:** secciones 8.2 y 8.7

En 2018 se conoció que una gran empresa tecnológica había abandonado una herramienta experimental de selección de personal porque penalizaba las hojas de vida de mujeres: había aprendido de los patrones de contratación de los diez años anteriores (Dastin, 2018; sección 8.2). En este cuaderno reproduces ese problema **en pequeño**, con datos **sintéticos**, y lo auditas con dos métricas sencillas.

Los datos son ficticios y se generan aquí mismo. Están construidos a propósito para mostrar cómo un modelo hereda el sesgo de sus datos de entrenamiento. Todas las celdas se probaron con Python 3 y solo usan `numpy`.

In [ ]:
import math
import random
from collections import Counter, defaultdict

import numpy as np

## 1. Un historial de contratación sesgado

Una empresa de software quiere automatizar el primer filtro de hojas de vida con un clasificador entrenado con sus **decisiones pasadas**. Cada candidatura tiene cuatro atributos:

| Atributo | Valores |
|---|---|
| `experiencia` | `baja`, `media`, `alta` |
| `prueba_tecnica` | resultado de una prueba técnica: `baja`, `media`, `alta` |
| `pausa_laboral` | ¿la hoja de vida tiene una pausa de más de un año? `si`, `no` |
| `genero` | `mujer`, `hombre` |

El **mérito real** de una candidatura depende solo de la experiencia y de la prueba técnica. Pero en el historial, a igual mérito, las mujeres fueron contratadas con menos frecuencia. Además, por razones sociales (por ejemplo, el cuidado de hijos o familiares), las pausas laborales son más frecuentes en las hojas de vida de mujeres, aunque no dicen nada sobre la capacidad técnica.

In [ ]:
def generar(n, semilla):
    rng = random.Random(semilla)
    datos = []
    for _ in range(n):
        genero = rng.choice(["mujer", "hombre"])
        experiencia = rng.choices(["baja", "media", "alta"], [0.4, 0.4, 0.2])[0]
        prueba = rng.choices(["baja", "media", "alta"], [0.3, 0.4, 0.3])[0]
        pausa = "si" if rng.random() < (0.45 if genero == "mujer" else 0.10) else "no"
        puntos = {"baja": 0, "media": 1, "alta": 2}
        merito = puntos[experiencia] + puntos[prueba] >= 2            # lo que debería decidir
        p_contratar = 0.85 if merito else 0.10
        if genero == "mujer":
            p_contratar *= 0.55                                        # sesgo histórico
        contratado = "si" if rng.random() < p_contratar else "no"
        datos.append({"experiencia": experiencia, "prueba_tecnica": prueba, "pausa_laboral": pausa,
                      "genero": genero, "merito": merito, "contratado": contratado})
    return datos


historial = generar(3000, semilla=1)
nuevas = generar(2000, semilla=2)                                      # candidaturas nuevas
for g in ["mujer", "hombre"]:
    grupo = [d for d in historial if d["genero"] == g]
    print(f"{g:<7} contratación histórica: {np.mean([d['contratado'] == 'si' for d in grupo]):.2f} | "
          f"proporción con mérito: {np.mean([d['merito'] for d in grupo]):.2f} | "
          f"con pausa laboral: {np.mean([d['pausa_laboral'] == 'si' for d in grupo]):.2f}")

Observa que la proporción de candidaturas con mérito es casi igual en los dos grupos, pero la tasa de contratación histórica no.

## 2. Entrenar un modelo con el historial

Usamos el clasificador bayesiano ingenuo de la semana 6, entrenado para imitar la columna `contratado`.

In [ ]:
class BayesIngenuoCategorico:
    def __init__(self, alfa=1.0):
        self.alfa = alfa

    def entrenar(self, X, y):
        self.clases = Counter(y)
        self.n = len(y)
        self.cuentas = defaultdict(Counter)
        self.valores = defaultdict(set)
        for x, c in zip(X, y):
            for i, v in enumerate(x):
                self.cuentas[(i, c)][v] += 1
                self.valores[i].add(v)
        return self

    def predecir(self, x):
        def puntaje(c):
            p = math.log(self.clases[c] / self.n)
            for i, v in enumerate(x):
                k = len(self.valores[i])
                p += math.log((self.cuentas[(i, c)][v] + self.alfa) / (self.clases[c] + self.alfa * k))
            return p
        return max(self.clases, key=puntaje)


def auditar(atributos, titulo):
    modelo = BayesIngenuoCategorico().entrenar([[d[a] for a in atributos] for d in historial],
                                              [d["contratado"] for d in historial])
    pred = {g: [] for g in ["mujer", "hombre"]}
    tvp = {g: [] for g in ["mujer", "hombre"]}
    for d in nuevas:
        seleccionado = modelo.predecir([d[a] for a in atributos]) == "si"
        pred[d["genero"]].append(seleccionado)
        if d["merito"]:
            tvp[d["genero"]].append(seleccionado)
    tasa = {g: np.mean(v) for g, v in pred.items()}
    print(f"\n{titulo}\n  atributos usados: {atributos}")
    for g in ["mujer", "hombre"]:
        print(f"  {g:<7} tasa de selección: {tasa[g]:.2f} | candidaturas con mérito seleccionadas: {np.mean(tvp[g]):.2f}")
    print(f"  Razón de impacto (mujer / hombre): {tasa['mujer'] / tasa['hombre']:.3f}"
          f"  -> {'supera' if tasa['mujer'] / tasa['hombre'] >= 0.8 else 'NO supera'} la regla de los cuatro quintos")


auditar(["experiencia", "prueba_tecnica", "pausa_laboral", "genero"], "Modelo A: con todos los atributos")

Usamos dos métricas (sección 8.2.3):

- **Tasa de selección por grupo** y su cociente, la **razón de impacto**. En Estados Unidos, una regla práctica usada en procesos de selección, la **regla de los cuatro quintos**, considera señal de posible discriminación que un grupo sea seleccionado a menos del 80 % de la tasa del otro.
- **Proporción de candidaturas con mérito que el modelo selecciona** en cada grupo (la tasa de verdaderos positivos respecto al mérito real). Si es distinta, el modelo trata de forma diferente a personas igual de calificadas.

## 3. «Quitemos el género y listo»

La reacción más común es eliminar el atributo sensible. Probémoslo.

In [ ]:
auditar(["experiencia", "prueba_tecnica", "pausa_laboral"], "Modelo B: sin el atributo género")

La brecha se reduce, pero **no desaparece**: el modelo sigue seleccionando menos mujeres con mérito. El atributo `pausa_laboral` está correlacionado con el género, y el modelo lo usa como **variable sustituta** (*proxy*) para reproducir el sesgo del historial.

In [ ]:
auditar(["experiencia", "prueba_tecnica"], "Modelo C: solo los atributos relacionados con el mérito")

### ✏️ Ejercicio. ¿Problema resuelto?

1. Compara los tres modelos. ¿Cuál cumple la regla de los cuatro quintos? ¿En cuál las candidaturas con mérito se seleccionan en proporciones parecidas en ambos grupos?
2. El modelo C aún se entrenó con etiquetas sesgadas (`contratado`). ¿Por qué, aun así, su comportamiento es más justo? ¿Qué pasaría si el mérito real dependiera también de un atributo correlacionado con el género?
3. En un caso real no conocemos el «mérito real». ¿Cómo podría una empresa detectar este problema antes de poner el modelo en producción? Menciona al menos dos prácticas de la sección 8.7.
4. ¿Debería automatizarse esta decisión? Argumenta tu respuesta con la clasificación de riesgos del Reglamento europeo de IA (sección 8.3.3).

*Tus respuestas:*

---

**Referencias**

Dastin, J. (2018, 10 de octubre). Amazon scraps secret AI recruiting tool that showed bias against women. *Reuters*. https://www.reuters.com/article/us-amazon-com-jobs-automation-insight-idUSKCN1MK08G